# Digits-5 matched graph-diagnostic pilot
Three methods, α=0.1, seed=42. Each method runs twice: exports off and on. Six executions verify that instrumentation leaves results unchanged. Original three-method training settings are retained; no hyperparameter tuning is performed.

In [ ]:
from pathlib import Path
import sys, subprocess, json, os

# Select the extracted directory containing pilot.py and icde/config.py.
PROJECT = Path.cwd().resolve()
candidates = [PROJECT, PROJECT / "FedGAPS_Diagnostic_Pilot"]
PROJECT = next((p for p in candidates if (p / "pilot.py").is_file() and (p / "icde/config.py").is_file()), PROJECT)
required = ["pilot.py", "icde/__init__.py", "icde/config.py", "reference/run_digits5.py"]
missing = [name for name in required if not (PROJECT / name).is_file()]
assert not missing, f"Incomplete extraction at {PROJECT}. Missing {missing}. Extract the entire ZIP into a fresh folder."
DATA_ROOT = PROJECT / "data"  # Change to the parent of your existing digits5 folder.
OUTPUT = PROJECT / "outputs/diagnostic_pilot_amp_bs64_w4"
DEVICE = "cuda"
RESUME = False
WORKERS = 4
BATCH_SIZE = 64
AMP = True

def run(*args):
    env = dict(os.environ)
    env["PYTHONPATH"] = str(PROJECT) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run([sys.executable, *map(str, args)], cwd=PROJECT, env=env, check=True)

print("Project:", PROJECT)
print("Python:", sys.executable)
run("-c", "import icde.config; print('Config loaded from:', icde.config.__file__)")


## Optional quick CPU check
This uses synthetic images and does not estimate Digits-5 performance. Use a fresh output directory, or add `--resume` after an interruption.

In [ ]:
# run("pilot.py", "--smoke", "--device", "cpu", "--output", PROJECT / "outputs/smoke_check")

## Matched real-data pilot
Uses ResNet-18, four clients per domain (20 total), SGD, 50 maximum rounds, up to 20 warm-up rounds, patience 5, batch size 64, and AMP disabled. Requires prepared Digits-5 data and pretrained weights. Runtime may increase with diagnostics; numerical results must match exactly.

In [ ]:
args = ["pilot.py", "--data-root", DATA_ROOT, "--output", OUTPUT, "--device", DEVICE, "--workers", str(WORKERS), "--batch-size", str(BATCH_SIZE)]
if AMP: args.append("--amp")
if RESUME: args.append("--resume")
run(*args)

In [ ]:
report = json.loads((OUTPUT / "verification.json").read_text())
assert len(report["checks"]) == 3 and all(x["exact_match"] for x in report["checks"])
report

## Plot graph diagnostics
The selected export is recomputed from the restored best warm-up checkpoint. Initial graphs are diagnostic only. The no-graph variant exports affinities for comparison, but does not use graph guidance. Node masks identify absent class/domain pairs; zero entries are not evidence of observed disconnection.

In [ ]:
run("plot_diagnostics.py", "--output", OUTPUT)

## Files to return
Share `verification.json`, the three on-run `diagnostics/` folders, and their `metrics.json`, `partition.json`, `membership.npy`, and `distance.npy`. Do not include raw data or large model checkpoints. Test probability exports are for exact verification and can remain local. Diagnostic graph files describe computed affinities, not independently optimized edge parameters.